In [20]:
import json
import numpy as np
import ollama
from sentence_transformers import SentenceTransformer

In [21]:
with open("../output/RO_NECP_chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
vectors = np.load("../output/RO_NECP_vectors.npy")

model=SentenceTransformer("BAAI/bge-small-en-v1.5")

LLM = "llama3.2"
PREFIX = "Represent this sentence for searching relevant passages: "

print(len(chunks), "chunks,", vectors.shape, "vectors")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

636 chunks, (636, 384) vectors


### retrieval

In [22]:
def retrieve(question):
    # embedding and return the most similar chunks
    q = model.encode(PREFIX + question, normalize_embeddings=True)
    top = np.argsort(vectors @ q)[::-1][:5]
    return [chunks[i] for i in top]

### prompt

In [27]:
def build_prompt(question, found):
    context = "\n\n".join(f"[page {c['page']}]\n{c['text']}" for c in found)
    return f"""You help people find data in national energy and climate plans.
Answer the question using ONLY the excerpts below.
Cite the page for every fact, like (p. 49)."

Excerpts:
{context}

Question: {question}
Answer:"""

### generation

In [28]:
def generate(prompt):
    reply = ollama.chat(
        model=LLM,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_ctx": 8192},
    )
    return reply["message"]["content"]

### try it out

In [31]:
def ask(question):
    found = retrieve(question)
    answer = generate(build_prompt(question, found))
    print("Q:", question)
    print("Retrieved pages:", [c["page"] for c in found])
    print("A:", answer, "\n")
    return answer
ask("Which law requires local authorities to separately collect municipal waste?")
ask("In which year will Romania reach climate neutrality?")

Q: Which law requires local authorities to separately collect municipal waste?
Retrieved pages: [83, 88, 84, 146, 49]
A: According to page 83, the law that requires local authorities to separately collect municipal waste is Law No 211/2011, as subsequently amended and supplemented, specifically Article 17(1). 

Q: In which year will Romania reach climate neutrality?
Retrieved pages: [48, 49, 11, 120, 13]
A: There is no information provided in the excerpts about Romania reaching climate neutrality in a specific year. The excerpts do provide information on Romania's greenhouse gas emission projections, energy consumption targets, and energy efficiency objectives, but do not mention climate neutrality. 



"There is no information provided in the excerpts about Romania reaching climate neutrality in a specific year. The excerpts do provide information on Romania's greenhouse gas emission projections, energy consumption targets, and energy efficiency objectives, but do not mention climate neutrality."